# Multi-Fidelity Gaussian Process Surrogates for Thermal PDE Systems

Surrogate modeling and cost-aware active learning for 2D steady-state heat diffusion using **GPyTorch**.

### Objectives:
1. **PDE Simulation:** Solve 2D heat diffusion under localized Gaussian heating sources via finite-difference relaxation.
2. **Multi-Fidelity Modeling:** Construct a cross-fidelity covariance kernel combining spatial Matérn-5/2 and task correlation ($k_{spatial} \otimes k_{task}$).
3. **Active Learning:** Dynamically select between low-fidelity (coarse, fast) and high-fidelity (fine, slow) simulations to minimize surrogate uncertainty under a strict compute budget.


## 1. Setup & Environment


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import gpytorch

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

os.makedirs('figures', exist_ok=True)


## 2. Finite-Difference Heat Diffusion Solver

We model steady-state heat conduction on a 2D plate $[0, 1] \times [0, 1]$ governed by the Poisson equation with localized heat generation:

$$\nabla^2 T(x, y) = -Q(x, y)$$

subject to Dirichlet boundary conditions ($T = 0$ along plate edges). Heat sources are modeled as Gaussian profiles centered at $(s_x, s_y)$ with heat power $P$. The discrete PDE is solved using Jacobi relaxation with a standard 5-point discrete Laplacian stencil.


In [ ]:
def simulate_heat(sources, grid_size=32, steps=100):
    """
    Finite-difference 2D steady-state heat diffusion simulation.
    sources: tensor of shape (N, 3) -> [[x1, y1, power1], [x2, y2, power2], ...]
    grid_size: spatial resolution (grid_size x grid_size)
    steps: number of Jacobi relaxation iterations
    """
    dx = 1.0 / (grid_size - 1)
    
    x = torch.linspace(0.0, 1.0, grid_size)
    y = torch.linspace(0.0, 1.0, grid_size)
    grid_y, grid_x = torch.meshgrid(y, x, indexing="ij")
    
    # Gaussian heat source injection
    Q = torch.zeros((1, 1, grid_size, grid_size))
    for src in sources:
        sx, sy, power = src[0], src[1], src[2]
        dist_sq = (grid_x - sx)**2 + (grid_y - sy)**2
        Q += power * torch.exp(-dist_sq / (2 * 0.08**2))
    
    # 5-point discrete Laplacian stencil for Jacobi iteration
    stencil = torch.tensor([
        [0.00, 0.25, 0.00],
        [0.25, 0.00, 0.25],
        [0.00, 0.25, 0.00]
    ]).view(1, 1, 3, 3)
    
    T = torch.zeros((1, 1, grid_size, grid_size))
    q_term = 0.25 * (dx ** 2) * Q
    
    # Relaxation loop
    for _ in range(steps):
        T = F.conv2d(T, stencil, padding=1) + q_term
        # Enforce cold Dirichlet boundaries
        T[:, :, 0, :] = 0.0
        T[:, :, -1, :] = 0.0
        T[:, :, :, 0] = 0.0
        T[:, :, :, -1] = 0.0
        
    return T.squeeze()

# Sample heater configuration
example_heaters = torch.tensor([
    [0.3, 0.4, 10.0],  # Heater 1: x=0.3, y=0.4, power=10
    [0.7, 0.7, 8.0]    # Heater 2: x=0.7, y=0.7, power=8
])

# Compare Low-Fidelity (16x16, 40 steps) vs High-Fidelity (64x64, 400 steps)
low_grid = simulate_heat(example_heaters, grid_size=16, steps=40)
high_grid = simulate_heat(example_heaters, grid_size=64, steps=400)

low_resampled = F.interpolate(
    low_grid.view(1, 1, 16, 16), size=(64, 64), mode='bilinear', align_corners=True
).squeeze()
diff_grid = torch.abs(high_grid - low_resampled)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im0 = axes[0].imshow(low_grid.numpy(), origin='lower', cmap='inferno', extent=[0, 1, 0, 1])
axes[0].scatter([0.3, 0.7], [0.4, 0.7], c='cyan', marker='x', s=80, label='Heaters')
axes[0].set_title(f"Low-Fidelity (16x16, 40 steps)\nPeak: {low_grid.max():.3f} °C")
fig.colorbar(im0, ax=axes[0], label="Temp (°C)")
axes[0].legend(loc="upper left")

im1 = axes[1].imshow(high_grid.numpy(), origin='lower', cmap='inferno', extent=[0, 1, 0, 1])
axes[1].scatter([0.3, 0.7], [0.4, 0.7], c='cyan', marker='x', s=80, label='Heaters')
axes[1].set_title(f"High-Fidelity (64x64, 400 steps)\nPeak: {high_grid.max():.3f} °C")
fig.colorbar(im1, ax=axes[1], label="Temp (°C)")

im2 = axes[2].imshow(diff_grid.numpy(), origin='lower', cmap='magma', extent=[0, 1, 0, 1])
axes[2].set_title(f"Absolute Discrepancy\nMax Diff: {diff_grid.max():.3f} °C")
fig.colorbar(im2, ax=axes[2], label="|Diff| (°C)")

for ax in axes:
    ax.set_xlabel("X Coordinate")
    ax.set_ylabel("Y Coordinate")

plt.tight_layout()
plt.savefig('figures/fidelity_comparison.png', dpi=200)
plt.show()


## 3. Computational Benchmark: Speed vs. Fidelity Trade-off


In [ ]:
def get_peak_temperature(sources, fidelity=0):
    start = time.perf_counter()
    if fidelity == 0:
        # Low-fidelity: coarse 16x16 grid, 40 steps
        grid = simulate_heat(sources, grid_size=16, steps=40)
    else:
        # High-fidelity: fine 64x64 grid, 400 steps
        grid = simulate_heat(sources, grid_size=64, steps=400)
    cost = time.perf_counter() - start
    peak_temp = grid.max().item()
    return peak_temp, cost

# Benchmark execution runtime on identical configuration
peak_fast, cost_fast = get_peak_temperature(example_heaters, fidelity=0)
peak_slow, cost_slow = get_peak_temperature(example_heaters, fidelity=1)

print(f"Low-Fidelity  (m=0): {peak_fast:.4f} °C | Runtime: {cost_fast*1000:.2f} ms")
print(f"High-Fidelity (m=1): {peak_slow:.4f} °C | Runtime: {cost_slow*1000:.2f} ms")
print(f"High-fidelity simulation is {cost_slow / cost_fast:.1f}x more computationally expensive.")


## 4. Multi-Fidelity Training Dataset

In multi-fidelity surrogate modeling, we leverage many cheap low-fidelity evaluations alongside a sparse set of high-fidelity evaluations. Each input vector contains 7 dimensions: 6 spatial features (2 heaters $\times$ $[x, y, P]$) + 1 fidelity indicator ($m \in \{0, 1\}$).


In [ ]:
def random_heater_setup():
    """Generates random coordinates and heat powers for 2 heating elements."""
    return torch.tensor([
        [np.random.uniform(0.2, 0.4), np.random.uniform(0.2, 0.8), np.random.uniform(5.0, 15.0)],
        [np.random.uniform(0.6, 0.8), np.random.uniform(0.2, 0.8), np.random.uniform(5.0, 15.0)]
    ], dtype=torch.float32)

X_rows = []
y_rows = []

# Collect 8 cheap low-fidelity runs
print("Generating 8 low-fidelity initial samples...")
for _ in range(8):
    setup = random_heater_setup()
    peak, _ = get_peak_temperature(setup, fidelity=0)
    row = torch.cat([setup.flatten(), torch.tensor([0.0])])
    X_rows.append(row)
    y_rows.append(peak)

# Collect 2 expensive high-fidelity runs
print("Generating 2 high-fidelity initial samples...")
for _ in range(2):
    setup = random_heater_setup()
    peak, _ = get_peak_temperature(setup, fidelity=1)
    row = torch.cat([setup.flatten(), torch.tensor([1.0])])
    X_rows.append(row)
    y_rows.append(peak)

train_X = torch.stack(X_rows)
train_y = torch.tensor(y_rows, dtype=torch.float32)

print(f"\nInitial dataset ready: {train_X.shape[0]} samples (8 low-fid, 2 high-fid)")
print("Input shape:", train_X.shape, "Target shape:", train_y.shape)


## 5. Multi-Fidelity Gaussian Process Model

The multi-task covariance kernel is defined as the product of a continuous spatial kernel over the physical design parameters and an index kernel over the discrete fidelity level:

$$K((x, m), (x', m')) = K_{\text{spatial}}(x, x') \cdot K_{\text{task}}(m, m')$$

- $K_{\text{spatial}}$: Matérn-5/2 kernel across the 6 physical parameters ($x_1, y_1, P_1, x_2, y_2, P_2$).
- $K_{\text{task}}$: Rank-1 IndexKernel learning the correlation matrix between low and high fidelity.


In [ ]:
class SimpleMultiFidelityGP(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        
        # Spatial distance kernel on physical heater parameters (features 0 through 5)
        self.spatial_kernel = gpytorch.kernels.MaternKernel(nu=2.5, active_dims=tuple(range(6)))
        
        # Cross-fidelity task correlation kernel on fidelity index (feature 6)
        self.task_kernel = gpytorch.kernels.IndexKernel(num_tasks=2, rank=1, active_dims=(6,))
        
        # Product kernel: Spatial * Task
        self.covar_module = self.spatial_kernel * self.task_kernel

    def forward(self, x):
        mean = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar)

likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = SimpleMultiFidelityGP(train_X, train_y, likelihood)

model.train()
likelihood.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

print("Optimizing Gaussian Process hyper-parameters...")
for step in range(80):
    optimizer.zero_grad()
    output = model(train_X)
    loss = -mll(output, train_y)
    loss.backward()
    optimizer.step()
    if (step + 1) % 20 == 0:
        print(f"Step {step+1:02d}/80 | Negative Marginal Log-Likelihood: {loss.item():.4f}")


## 6. Surrogate Validation & Parity Analysis

We evaluate the trained multi-fidelity GP by predicting the high-fidelity peak temperature across unseen test configurations and comparing predictions against true high-fidelity numerical solutions.


In [ ]:
# Evaluate surrogate on a single test configuration
new_test_setup = random_heater_setup()
test_input = torch.cat([new_test_setup.flatten(), torch.tensor([1.0])]).unsqueeze(0)

model.eval()
likelihood.eval()

with torch.no_grad():
    prediction_dist = likelihood(model(test_input))
    pred_mean = prediction_dist.mean.item()
    pred_std = prediction_dist.stddev.item()

actual_temp, _ = get_peak_temperature(new_test_setup, fidelity=1)

print("=" * 55)
print(f"Surrogate Posterior: {pred_mean:.4f} °C ± {2*pred_std:.4f} °C (95% CI)")
print(f"Ground Truth (HF):  {actual_temp:.4f} °C")
print(f"Absolute Error:     {abs(pred_mean - actual_temp):.4f} °C")
print("=" * 55)

# Generate Parity Plot across 10 random test configurations
test_setups = [random_heater_setup() for _ in range(10)]
true_vals, p_means, p_stds = [], [], []

with torch.no_grad():
    for s in test_setups:
        inp = torch.cat([s.flatten(), torch.tensor([1.0])]).unsqueeze(0)
        dist = likelihood(model(inp))
        p_means.append(dist.mean.item())
        p_stds.append(dist.stddev.item())
        t_val, _ = get_peak_temperature(s, fidelity=1)
        true_vals.append(t_val)

fig, ax = plt.subplots(figsize=(6, 5))
ax.errorbar(true_vals, p_means, yerr=2*np.array(p_stds), fmt='o', color='#1f77b4',
            ecolor='#aec7e8', elinewidth=2, capsize=4, label='Surrogate Posterior (±2σ)')
min_v = min(min(true_vals), min(p_means)) - 0.2
max_v = max(max(true_vals), max(p_means)) + 0.2
ax.plot([min_v, max_v], [min_v, max_v], 'k--', lw=1.5, label='Ideal 1:1 Reference')
ax.set_xlabel('Ground Truth High-Fidelity Peak Temp (°C)')
ax.set_ylabel('Surrogate Predicted Peak Temp (°C)')
ax.set_title('Multi-Fidelity GP Validation Parity Plot')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/surrogate_parity.png', dpi=200)
plt.show()


## 7. Cost-Aware Acquisition Policy

Active learning chooses whether to query a cheap, coarse simulation or an expensive, fine simulation by balancing epistemic uncertainty against execution cost:

$$\alpha(x, m) = \frac{\sigma_m^2(x)}{c_m}$$

where $\sigma_m^2(x)$ is the posterior variance at fidelity $m$, and $c_m$ is the computational cost.


In [ ]:
candidates = [random_heater_setup().flatten() for _ in range(10)]
cand_tensor = torch.stack(candidates)

# Evaluate uncertainty for both low (cost ~ 1) and high (cost ~ 30) fidelities
with torch.no_grad():
    cand_low = torch.cat([cand_tensor, torch.zeros(10, 1)], dim=-1)
    cand_high = torch.cat([cand_tensor, torch.ones(10, 1)], dim=-1)
    
    uncert_low = likelihood(model(cand_low)).variance
    uncert_high = likelihood(model(cand_high)).variance

cost_low = 1.0
cost_high = 30.0

value_low = uncert_low / cost_low
value_high = uncert_high / cost_high

print("Candidate Evaluation (Information Gain / Compute Cost):")
print(f"Max Value (Low-Fidelity Query):  {value_low.max().item():.5f}")
print(f"Max Value (High-Fidelity Query): {value_high.max().item():.5f}")

if value_low.max() > value_high.max():
    print("\nDecision: Acquire LOW-FIDELITY simulation for maximum compute efficiency.")
else:
    print("\nDecision: Acquire HIGH-FIDELITY simulation to resolve high variance.")


## 8. Multi-Fidelity Active Learning Loop & Convergence Tracking

We execute an automated active learning cycle for 5 iterations. At each step:
1. Propose candidate design points.
2. Score acquisition utility $\alpha(x, m)$ across fidelities.
3. Query the selected simulation fidelity.
4. Retrain the multi-fidelity GP.
5. Track posterior variance reduction and cumulative simulation runtime.


In [ ]:
print("Starting Multi-Fidelity Active Learning Loop...\n")

history_iters = [0]
history_time = [0.0]
history_fids = [None]

# Pool of holdout points to monitor posterior variance reduction
eval_pool = torch.stack([random_heater_setup().flatten() for _ in range(20)])
eval_high = torch.cat([eval_pool, torch.ones(20, 1)], dim=-1)

model.eval()
likelihood.eval()
with torch.no_grad():
    init_var = likelihood(model(eval_high)).variance.mean().item()
history_var = [init_var]

total_time_spent = 0.0

for loop_iter in range(1, 6):
    # 1. Propose 20 candidate configurations
    candidates = [random_heater_setup().flatten() for _ in range(20)]
    cand_tensor = torch.stack(candidates)
    
    # 2. Score candidates across both fidelities
    model.eval()
    likelihood.eval()
    with torch.no_grad():
        c_low = torch.cat([cand_tensor, torch.zeros(20, 1)], dim=-1)
        c_high = torch.cat([cand_tensor, torch.ones(20, 1)], dim=-1)
        
        val_low = likelihood(model(c_low)).variance / 17.0   # Coarse cost ~17ms
        val_high = likelihood(model(c_high)).variance / 70.0 # Fine cost ~70ms
    
    max_val_low, idx_low = val_low.max(dim=0)
    max_val_high, idx_high = val_high.max(dim=0)
    
    if max_val_low >= max_val_high:
        chosen_fidelity = 0
        chosen_setup = candidates[idx_low]
    else:
        chosen_fidelity = 1
        chosen_setup = candidates[idx_high]
        
    # 3. Query the PDE solver
    sources = chosen_setup.view(2, 3)
    new_temp, runtime = get_peak_temperature(sources, fidelity=chosen_fidelity)
    total_time_spent += runtime
    
    fid_name = "LOW (m=0)" if chosen_fidelity == 0 else "HIGH (m=1)"
    print(f"Iteration {loop_iter}: Selected {fid_name} | Sim time: {runtime*1000:.1f} ms | Peak: {new_temp:.4f} °C")
    
    # 4. Augment training set
    new_x = torch.cat([chosen_setup, torch.tensor([float(chosen_fidelity)])]).unsqueeze(0)
    train_X = torch.cat([train_X, new_x], dim=0)
    train_y = torch.cat([train_y, torch.tensor([new_temp])], dim=0)
    
    # 5. Retrain multi-fidelity GP
    model = SimpleMultiFidelityGP(train_X, train_y, likelihood)
    model.train()
    likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.08)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)
    
    for _ in range(40):
        optimizer.zero_grad()
        loss = -mll(model(train_X), train_y)
        loss.backward()
        optimizer.step()
        
    # 6. Evaluate uncertainty reduction
    model.eval()
    likelihood.eval()
    with torch.no_grad():
        cur_var = likelihood(model(eval_high)).variance.mean().item()
        
    history_iters.append(loop_iter)
    history_time.append(total_time_spent * 1000)
    history_fids.append('Low' if chosen_fidelity == 0 else 'High')
    history_var.append(cur_var)

print(f"\nActive Learning complete! Total compute spent: {total_time_spent*1000:.1f} ms")
print(f"Total training set size: {len(train_X)} samples")

# Plot Convergence Figures
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

ax1.plot(history_iters, history_var, marker='o', color='#2ca02c', lw=2, label='Mean Posterior Variance')
ax1.set_ylabel('High-Fidelity Variance')
ax1.set_title('Cost-Aware Active Learning Convergence')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(history_iters, history_time, marker='s', color='#d62728', lw=2, label='Cumulative Sim Time (ms)')
for i in range(1, len(history_iters)):
    ax2.annotate(f'Query: {history_fids[i]}', (history_iters[i], history_time[i]),
                 textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax2.set_xlabel('Active Learning Iteration')
ax2.set_ylabel('Compute Budget Spent (ms)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('figures/active_learning_progress.png', dpi=200)
plt.show()
